# Complex Ginzburg--Landau equation on a cow surface

Semi-implicit surface reaction-diffusion experiment with a reusable surface solution operator.


In [ ]:
from pathlib import Path
import sys
import time

project = Path.cwd().resolve()
if project.name == 'notebooks':
    project = project.parent
sys.path.insert(0, str(project))

import numpy as np
import pysurfacefun as psf

output_dir = project / 'notebook_outputs'
output_dir.mkdir(exist_ok=True)

## Parameters

This is a complex-valued scalar equation.  The diffusion part is treated implicitly through `(I - dt delta Delta_Gamma)`, while the cubic reaction is explicit.

In [ ]:
delta = 5e-4
c = 1.5
dt = 0.03

def N(u):
    return u - (1.0 + c * 1j) * u * (abs(u) ** 2)

## Load the Cow Geometry

`cow.csv` has `339` patches, each stored as `8 x 8` Rhino samples.  Use `solve_n = 12` for the high-order run; for a quick run, `solve_n = 8` is already close to the imported geometry.

In [ ]:
cow_file = project / 'notebook_data' / 'cow.csv'
rhino_n = 8
solve_n = 8     # use 12 for the high-order run

dom = psf.from_rhino(str(cow_file), rhino_n)
if solve_n != rhino_n:
    dom = psf.resample_mesh(dom, solve_n)

print('patches:', dom.npatches)
print('n:', dom.n, 'order:', dom.order)
print('bounding box:', psf.boundingbox(dom))

## Initial Data

`psf.randnfun3` gives a reproducible smooth random Fourier field over the geometry bounding box.

In [ ]:
bb = psf.boundingbox(dom)
u_noise = psf.randnfun3(0.2, bb, seed=1)

u = psf.surfacefun(lambda x, y, z: u_noise(x, y, z), dom)

print('initial real(u) range:', psf.minEst(psf.real(u)), psf.maxEst(psf.real(u)))

## Build Reusable Operators

The expensive part is the construction of the surface solution operators.  After this, `update_rhs` reuses the cached inverse matrices and only pushes the new particular solution through the merge tree.

In [ ]:
t0 = time.perf_counter()
L = psf.surfaceop(dom, {'lap': -dt * delta, 'b': 1.0}, 0.0)
L.build()
print(f'build time: {time.perf_counter() - t0:.2f} s')

## Time Stepping

The semi-implicit Euler update is

\[
(I-dt\,\delta\Delta_\Gamma)u^{k+1}=u^k+dt\,\left[u^k-(1+ci)u^k|u^k|^2\right].
\]

In [ ]:
kend = 2000
print_every = 10

for k in range(1, kend + 1):
    t0 = time.perf_counter()
    u = L.apply(u + dt * N(u))
    if k == 1 or k == kend or k % print_every == 0:
        real_u = psf.real(u)
        print(
            f'k = {k:4d}, step time = {time.perf_counter() - t0:.2f} s, '
            f'real(u) range = [{psf.minEst(real_u):.3e}, {psf.maxEst(real_u):.3e}], '
            f'|u| max = {psf.maxEst(abs(u)):.3e}'
        )

## Visualization / Export

If `matplotlib` is installed, plot the final `u`.  If `meshio` is installed, export a VTU file for ParaView.

In [ ]:
try:
    psf.plot_surface(psf.real(u), title='cow complex Ginzburg-Landau: real(u)', vmin=-1.0, vmax=1.0)
except Exception as exc:
    print('Matplotlib plot skipped:', exc)

try:
    psf.write_vtu(output_dir / 'cow_complex_ginzburg_landau_real_u.vtu', psf.real(u), point_name='real_u')
    print('wrote cow_complex_ginzburg_landau_real_u.vtu')
except Exception as exc:
    print('VTU export skipped:', exc)